# 03 · Cross-encoder: baseline B3 (Amazon MiniLM reproduction)

Fine-tunes `cross-encoder/ms-marco-MiniLM-L-12-v2` on query + product title with MSE on the gains, using Amazon's settings, and scores val and test. The same notebook is the starting point for Direction 1 (input fields, DeBERTa) and Direction 2 (losses).

**Kaggle settings:** Accelerator **GPU T4** (or P100), **Internet on**.

**Inputs:**
- Dataset `wm-src` (the repo's `src/`, see README). Required.
- Add Input → Notebook Output → **preprocess notebook** (`pairs.parquet`, `products_clean.parquet`). Required.
- Add Input → Notebook Output → **bm25 notebook**. Optional: enables the paired comparison with BM25.

**How to run:**
1. `MODE = 'smoke'` (5% of train queries, 1 seed, plus the GPU speed benchmark). Run interactively, about 10–15 minutes. Check the smoke-test verdict at the end of section 3.
2. `MODE = 'full'` (all train queries, seeds 42 and 43), then **Save Version → Save & Run All**. Estimated under 1 hour; the training log prints the real speed.

**Settings from Amazon's `ranking/train.py`:** MiniLM-L12 (MS MARCO), `num_labels = 1` with no activation, MSE on gains E 1 / S 0.1 / C 0.01 / I 0, title only, batch 32, 1 epoch, lr 7e-6, 5,000 warm-up steps with linear decay, AdamW (weight decay 0.01, gradient clipping 1.0).

**Differences from Amazon:**
- `max_len` is 128 instead of 512. Query + title almost never reaches 128 tokens; section 1 prints the truncated share.
- The model trains on our train split (18,799 queries). Amazon trained on the official train minus 400 queries, about 10% more data.
- fp16 mixed precision, for speed on T4.

**Output** (`/kaggle/working/results/`): `<run>.json` and `scores/<run>_{val,test}.parquet`, in the same format as `02_bm25`, plus `gpu_benchmark.json` in smoke mode.

In [1]:
# Setup: make src/ importable. On Kaggle, attach the dataset `wm-src` (zip of src/, see README).
import glob, sys
from pathlib import Path

_hits = glob.glob('/kaggle/input/**/src/evaluation.py', recursive=True) + glob.glob('../src/evaluation.py')
assert _hits, 'src/ not found: Add Input -> dataset wm-src (zip of src/, see README)'
ROOT = Path(_hits[0]).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.evaluation import src_version
print('src from', ROOT, '|', src_version())

src from /kaggle/input/datasets/dchromevn/wm-src | {'files_sha1': '6172f3a190'}


In [2]:
import gc, json, os, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from src.cross_encoder import encode, make_optimizer, make_scaler, predict, train_ce
from src.data import load_pairs, product_texts
from src.evaluation import RunLog

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
transformers.logging.set_verbosity_error()

MODE = 'smoke'   # 'smoke': 5% of train queries, seed 42, + GPU benchmark | 'full': all train queries, seeds 42 and 43

CFG = {
    'model': 'cross-encoder/ms-marco-MiniLM-L-12-v2',
    'fields': 'title',        # 'title' | 'title_bc' | 'main' (Direction 1)
    'max_len': 128,
    'loss': 'mse',            # Direction 2 adds 'ce4' and 'listnet'
    'epochs': 1,
    'lr': 7e-6,
    'batch_size': 32,
    'warmup_steps': 5000,
    'weight_decay': 0.01,
    'max_grad_norm': 1.0,
    'fp16': True,
}
SEEDS = [42] if MODE == 'smoke' else [42, 43]
SUBSAMPLE = 0.05 if MODE == 'smoke' else 1.0    # share of train queries
EVAL_EVERY = 100 if MODE == 'smoke' else 2000   # steps between val checks (monitoring only, no early stopping)
EVAL_SPLITS = ['val'] if MODE == 'smoke' else ['val', 'test']
BENCHMARK = MODE == 'smoke'
SAVE_MODEL = False

SEED = 42
OUT = Path('/kaggle/working')
RES = OUT / 'results'
(RES / 'scores').mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), 'No GPU: Settings -> Accelerator -> GPU T4'
DEVICE = 'cuda'
print('torch', torch.__version__, '| transformers', transformers.__version__, '|', torch.cuda.get_device_name(0))
print('MODE', MODE, '| seeds', SEEDS, '| subsample', SUBSAMPLE)

torch 2.11.0+cu130 | transformers 5.20.2 | Tesla T4
MODE smoke | seeds [42] | subsample 0.05


## 1. Data

In [3]:
pairs = load_pairs(CFG['fields'])   # pairs.parquet + `text` for the preset (src/data.py)
MAIN_TEXT = product_texts('main') if BENCHMARK else None   # only for the GPU benchmark

MASK = {s: (pairs['split'] == s).to_numpy() for s in ['val', 'test']}
train_all = pairs[pairs['split'] == 'train']
if SUBSAMPLE < 1:
    keep = train_all['query_id'].drop_duplicates().sample(frac=SUBSAMPLE, random_state=SEED)
    train = train_all[train_all['query_id'].isin(keep)].reset_index(drop=True)
else:
    train = train_all.reset_index(drop=True)
print(f'train {train["query_id"].nunique():,} queries / {len(train):,} pairs | '
      f'val {MASK["val"].sum():,} pairs | test {MASK["test"].sum():,} pairs | fields = {CFG["fields"]}')

tok = AutoTokenizer.from_pretrained(CFG['model'])
smp = pairs.sample(20_000, random_state=SEED)
n_tok = np.array([len(x) for x in tok(smp['query'].tolist(), smp['text'].tolist())['input_ids']])
print(f'query + text tokens: median {np.median(n_tok):.0f}, p99 {np.percentile(n_tok, 99):.0f}, '
      f'max {n_tok.max()} | truncated at max_len {CFG["max_len"]}: {(n_tok > CFG["max_len"]).mean():.2%}')

train 940 queries / 18,674 pairs | val 41,513 pairs | test 181,701 pairs | fields = title


config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

query + text tokens: median 33, p99 70, max 126 | truncated at max_len 128: 0.00%


## 2. Training and prediction

The code is in `src/cross_encoder.py` (`train_ce`, `predict`, `compute_loss`); a new loss for Direction 2 goes into `compute_loss` there.

- Plain PyTorch loop with the Hugging Face model (`num_labels = 1`, raw logit as the score), so Direction 2 only needs a new loss.
- Batches are shuffled pairs (seeded). Pairs are tokenized on the fly and padded to the longest pair in the batch.
- Prediction sorts pairs by length to reduce padding, then puts the scores back in the original order.
- Every `EVAL_EVERY` steps the full val set is scored, only to watch the curve. The model kept is always the one after the last step, as in Amazon's script.

## 3. Logging runs, zero-shot model and B3

`RunLog` (`src/evaluation.py`) writes the same files as in `02_bm25`, plus `src_version` and library versions in each json. If the bm25 notebook output is attached, every run is also compared with `b1_bm25_main` by paired bootstrap.

The **zero-shot** run scores pairs with the MS MARCO model before any fine-tuning. It is the starting point of B3 and shows how much fine-tuning on ESCI adds.

In [4]:
def split_part(split):
    return pairs.loc[MASK[split], ['example_id', 'query_id', 'query', 'text', 'esci_label']]


log = RunLog(RES, {s: split_part(s) for s in EVAL_SPLITS})


def log_run(name, scores, info, compare_to=None):
    return log.log(name, scores, {'mode': MODE, **info}, compare_to=compare_to)


REF = 'b1_bm25_main' if log.load('b1_bm25_main') else None
print('paired comparison with', REF or 'nothing (bm25 notebook output not attached)')

b1_bm25_main val: nDCG 0.8227
paired comparison with b1_bm25_main


In [5]:
model0 = AutoModelForSequenceClassification.from_pretrained(CFG['model'], num_labels=1).float().to(DEVICE)
zs = {s: predict(model0, tok, split_part(s), CFG['max_len'], fp16=CFG['fp16']) for s in EVAL_SPLITS}
log_run(('smoke_' if MODE == 'smoke' else '') + f"b3_minilm_{CFG['fields']}_zeroshot", zs,
        {'model': CFG['model'], 'fields': CFG['fields'], 'max_len': CFG['max_len'], 'fine_tuned': False}, compare_to=REF)
del model0
torch.cuda.empty_cache()

model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

smoke_b3_minilm_title_zeroshot {'val': (0.833, [0.827, 0.8388], 'E>S', 0.6262)}


In [6]:
val_df = split_part('val')
b3_runs = []
for seed in SEEDS:
    name = f"{'smoke_' if MODE == 'smoke' else ''}b3_minilm_{CFG['fields']}_{CFG['loss']}_s{seed}"
    print(f'==== {name}')
    model, tokenizer, info = train_ce(CFG, train, seed, eval_df=val_df, eval_every=EVAL_EVERY,
                                     print_every=100 if MODE == 'smoke' else 1000)
    scores = {s: predict(model, tokenizer, split_part(s), CFG['max_len'], fp16=CFG['fp16']) for s in EVAL_SPLITS}
    log_run(name, scores, {**CFG, 'seed': seed, 'subsample': SUBSAMPLE,
                           'train_queries': int(train['query_id'].nunique()), 'train_pairs': len(train),
                           'gpu': torch.cuda.get_device_name(0), **info}, compare_to=REF)
    b3_runs.append(name)
    if SAVE_MODEL:
        model.save_pretrained(OUT / f'model_{name}')
        tokenizer.save_pretrained(OUT / f'model_{name}')
    del model
    torch.cuda.empty_cache()

==== smoke_b3_minilm_title_mse_s42


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/kaggle/input/datasets/dchromevn/wm-src/src/cross_encoder.py:104: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()


step    100/583 | loss 36.11480 | 17.3 it/s | 0.1 min, ~0 min left
   val @ 100: nDCG 0.8342 | E-over-S 0.6277
step    200/583 | loss 8.01070 | 18.5 it/s | 0.2 min, ~0 min left
   val @ 200: nDCG 0.7784 | E-over-S 0.5614
step    300/583 | loss 0.28392 | 18.8 it/s | 0.3 min, ~0 min left
   val @ 300: nDCG 0.8169 | E-over-S 0.6087
step    400/583 | loss 0.22967 | 19.1 it/s | 0.3 min, ~0 min left
   val @ 400: nDCG 0.8247 | E-over-S 0.6183
step    500/583 | loss 0.22191 | 19.1 it/s | 0.4 min, ~0 min left
   val @ 500: nDCG 0.8276 | E-over-S 0.6211
step    583/583 | loss 0.22310 | 19.1 it/s | 0.5 min, ~0 min left
   val @ 583: nDCG 0.8286 | E-over-S 0.6227
smoke_b3_minilm_title_mse_s42 {'val': (0.8286, [0.8221, 0.835], 'E>S', 0.6227)}


In [7]:
if MODE == 'smoke':
    r = log.records[b3_runs[0]]
    zs_name = f"smoke_b3_minilm_{CFG['fields']}_zeroshot"
    first, last = r['loss_log'][0]['loss'], r['loss_log'][-1]['loss']
    checks = {
        'loss goes down': last < first,
        'val nDCG >= zero-shot': r['metrics']['val']['ndcg'] >= log.records[zs_name]['metrics']['val']['ndcg'],
        'val nDCG > BM25 title (0.816)': r['metrics']['val']['ndcg'] > 0.816,
    }
    print(f'loss {first:.5f} -> {last:.5f} | speed {r["it_per_s"]} it/s -> '
          f'full epoch ~{len(train_all) / CFG["batch_size"] / r["it_per_s"] / 60:.0f} min per seed')
    for k, v in checks.items():
        print(('OK    ' if v else 'CHECK ') + k)
    print('Smoke test passed: switch MODE to full.' if all(checks.values()) else
          'Some checks failed: send me this output before the full run.')

loss 36.11480 -> 0.22310 | speed 19.15 it/s -> full epoch ~10 min per seed
OK    loss goes down
CHECK val nDCG >= zero-shot
OK    val nDCG > BM25 title (0.816)
Some checks failed: send me this output before the full run.


## 4. Results

`ndcg_amazon` on test is the number to compare with Amazon's 0.857. Their training set was about 10% larger, so a gap of about 0.005 is not a failure to reproduce.

In [8]:
display(log.table(('ndcg', 'ndcg_ci95', 'ndcg_amazon', 'ndcg10', 'mrr_e', 'e_over_s')))

if len(b3_runs) > 1:
    for s in EVAL_SPLITS:
        v = {m: [log.records[n]['metrics'][s][m] for n in b3_runs] for m in ['ndcg', 'ndcg_amazon', 'e_over_s']}
        print(s, {m: f'mean {np.mean(x):.4f} (seeds {x})' for m, x in v.items()})
    print('Amazon reference (test, amazon_script gains): 0.857')

log.print_paired()

ndcg        ndcg_ci95  ndcg_amazon  \
split run                                                                    
val   smoke_b3_minilm_title_mse_s42   0.8286  [0.8221, 0.835]       0.8067   
      smoke_b3_minilm_title_zeroshot  0.8330  [0.827, 0.8388]       0.8124   

                                      ndcg10   mrr_e  e_over_s  
split run                                                       
val   smoke_b3_minilm_title_mse_s42   0.7085  0.7765    0.6227  
      smoke_b3_minilm_title_zeroshot  0.7163  0.7868    0.6262

val  smoke_b3_minilm_title_zeroshot - b1_bm25_main: {'diff': 0.0103, 'ci95': [0.0055, 0.0156], 'p_b_le_a': 0.0}
val  smoke_b3_minilm_title_mse_s42 - b1_bm25_main: {'diff': 0.0058, 'ci95': [0.0003, 0.0114], 'p_b_le_a': 0.018}


## 5. GPU benchmark (smoke mode only)

Times 100 training steps (after 20 warm-up steps) for the configurations planned in Direction 1, using real `main` or `title` text. `hours_per_epoch` is for the whole train split (378,140 pairs, batch 32). The numbers decide `max_len` for DeBERTa and whether out-of-fold training for Direction 3 uses 5 or 2 folds (rule: 5 folds if one DeBERTa epoch takes at most about 8 GPU hours).

In [9]:
def benchmark(model_name, fields, max_len, batch_size=32, n_steps=100, warm=20):
    smp = pairs[pairs['split'] == 'train'].sample(batch_size * (n_steps + warm), random_state=SEED)
    assert fields in (CFG['fields'], 'main')
    q = smp['query'].to_numpy()
    t = (smp['text'] if fields == CFG['fields'] else smp['product_id'].map(MAIN_TEXT)).to_numpy()
    y = torch.from_numpy(smp['gain'].to_numpy(np.float32)).to(DEVICE)
    row = {'model': model_name, 'fields': fields, 'max_len': max_len, 'batch_size': batch_size}
    model = opt = None
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1).float().to(DEVICE)
        cfg = {**CFG, 'model': model_name}
        opt, scaler = make_optimizer(model, cfg), make_scaler(True)
        torch.cuda.reset_peak_memory_stats()
        finite = True
        for i in range(warm + n_steps):
            if i == warm:
                torch.cuda.synchronize()
                t0 = time.time()
            sl = slice(i * batch_size, (i + 1) * batch_size)
            with torch.autocast('cuda', dtype=torch.float16):
                pred = model(**encode(tokenizer, q[sl], t[sl], max_len)).logits.squeeze(-1)
            loss = F.mse_loss(pred.float(), y[sl])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            finite &= bool(torch.isfinite(loss))
        torch.cuda.synchronize()
        sec = (time.time() - t0) / n_steps
        row.update(sec_per_step=round(sec, 4), hours_per_epoch=round(sec * len(train_all) / batch_size / 3600, 2),
                   peak_gb=round(torch.cuda.max_memory_allocated() / 1e9, 1), loss_finite=finite)
    except torch.cuda.OutOfMemoryError:
        row['error'] = 'out of memory'
    finally:
        del model, opt
        gc.collect()
        torch.cuda.empty_cache()
    print(row)
    return row


if BENCHMARK:
    bench = [benchmark(m, f, L) for m, f, L in [
        ('cross-encoder/ms-marco-MiniLM-L-12-v2', 'title', 128),
        ('cross-encoder/ms-marco-MiniLM-L-12-v2', 'main', 256),
        ('microsoft/deberta-v3-base', 'title', 128),
        ('microsoft/deberta-v3-base', 'main', 192),
        ('microsoft/deberta-v3-base', 'main', 256),
    ]]
    bench_df = pd.DataFrame(bench)
    display(bench_df)
    (RES / 'gpu_benchmark.json').write_text(json.dumps(
        {'gpu': torch.cuda.get_device_name(0), 'created_utc': time.strftime('%Y-%m-%d %H:%M', time.gmtime()),
         'rows': bench}, indent=2))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'model': 'cross-encoder/ms-marco-MiniLM-L-12-v2', 'fields': 'title', 'max_len': 128, 'batch_size': 32, 'sec_per_step': 0.055, 'hours_per_epoch': 0.18, 'peak_gb': 1.3, 'loss_finite': True}


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

{'model': 'cross-encoder/ms-marco-MiniLM-L-12-v2', 'fields': 'main', 'max_len': 256, 'batch_size': 32, 'sec_per_step': 0.1529, 'hours_per_epoch': 0.5, 'peak_gb': 2.3, 'loss_finite': True}


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

... (9 outputs left out)

{'model': 'microsoft/deberta-v3-base', 'fields': 'main', 'max_len': 192, 'batch_size': 32, 'sec_per_step': 0.4893, 'hours_per_epoch': 1.61, 'peak_gb': 9.4, 'loss_finite': True}


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

{'model': 'microsoft/deberta-v3-base', 'fields': 'main', 'max_len': 256, 'batch_size': 32, 'sec_per_step': 0.6604, 'hours_per_epoch': 2.17, 'peak_gb': 11.7, 'loss_finite': True}


,model,fields,max_len,batch_size,sec_per_step,hours_per_epoch,peak_gb,loss_finite
0,cross-encoder/ms-marco-MiniLM-L-12-v2,title,128,32,0.0550,0.18,1.3,True
1,cross-encoder/ms-marco-MiniLM-L-12-v2,main,256,32,0.1529,0.50,2.3,True
2,microsoft/deberta-v3-base,title,128,32,0.2103,0.69,6.1,True
3,microsoft/deberta-v3-base,main,192,32,0.4893,1.61,9.4,True
4,microsoft/deberta-v3-base,main,256,32,0.6604,2.17,11.7,True
